In [ ]:
query = f"""
select
*
from Alice.prospectos_correos_alfin
where fecha_registro >='2026-08-01'
and estado='ENVIADO'
"""
df_correo = pd.read_sql(query, engine_mysql)
query = f"""
select
dni_cliente, fecha_visita,
DATE(fecha_envio) AS fecha_envio
from Alice.prospectos_envio_alfin
where fecha_creacion >='2026-08-01'
and estado='PROCESADO'
"""
df_formulario = pd.read_sql(query, engine_mysql)


In [37]:
display(df_correo.head(2))
display(df_formulario.head(2))

,dni_cliente,nombre_cliente,celular,fecha_visita,hora_visita,fecha_envio
0,48124718,CHOQUE CURO NOEMY LUZMERY,937616610,2026-08-04,0 days 11:30:00,2026-08-01
1,41494656,GUTIERREZ HEREDIA CARLOS ALBERTO,994746090,2026-08-06,0 days 12:00:00,2026-08-01


,dni_cliente,fecha_visita,fecha_envio
0,80644018,2026-08-05,2026-08-02
1,80632938,2026-08-02,2026-08-01


In [ ]:
df_correo=df_correo.merge(df_formulario,on=['dni_cliente','fecha_envio'],how='inner')

In [42]:
df_group = (
    df_correo
    .groupby(["fecha_envio"])
    .size()
    .reset_index(name="cantidad")
)
df_group.head()

,fecha_envio,cantidad
0,2026-08-01,2206
1,2026-08-02,2714


In [19]:
print(df_formulario.columns.tolist())

['id', 'hash_duplicado', 'dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion', 'estado', 'codigo_http_ms', 'respuesta_ms', 'fecha_creacion', 'fecha_envio']


In [43]:

ruta_archivo = os.path.join(ruta_alfin, 'usar_01.csv')
df_correo.to_csv(ruta_archivo, sep=';')

## inicio

In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
from funciones import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

    

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
filename='para_envio_alfin_01.csv'
df_lista=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db2.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db2.csv'
df_validar_03=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db.csv'
df_validar_04=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar_01=df_validar_01.drop('TASA_MIN_DESCUENTO')
df_validar_02=df_validar_02.drop('TASA_MIN_DESCUENTO')
df_validar_01=df_validar_01.withColumn('tipo_archivo',F.lit('campo'))
df_validar_02=df_validar_02.withColumn('tipo_archivo',F.lit('campo'))
df_validar_03=df_validar_03.withColumn('tipo_archivo',F.lit('call'))
df_validar_04=df_validar_04.withColumn('tipo_archivo',F.lit('call'))

df_validar=df_validar_01.unionByName(df_validar_02).unionByName(df_validar_03).unionByName(df_validar_04)


In [4]:
df_validar.filter(F.col('DNI')=='41844714').show()

+--------+---------------+-----------+--------+-------------+-------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+-------------+----------------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+-----------+------------+
|     DNI|    COLOR_FINAL|COD_USER_V3| USER_V3|    PERFIL_RO|campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|   GRUPO_TASA|       TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA_1_SS|TASA_2_SS|TASA_3_SS|TASA_4_SS|TASA_5_SS|TASA_6_SS|TASA_7_SS|ALERTA_MAQUETA| FEN|PERFIL_ESPECIAL|       TIPO|tipo_archivo|
+--------+---------------+-----------+--------+-------

In [ ]:
print(df_validar.columns)
print(df_lista.columns)



['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']
['dni', 'nombre', 'celular', 'agencia', 'hoja']


In [3]:
df_lista_1=df_lista.join(df_validar,['DNI'],'inner')
df_lista_1=df_lista_1.dropDuplicates(['DNI'])
print(df_lista_1.columns)


['dni', 'nombre', 'celular', 'agencia', 'hoja', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']


In [8]:
print(df_lista_1.columns)


['dni', 'nombre', 'celular', 'agencia', 'hoja', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']


In [4]:
from pyspark.sql.window import Window

# Ventana por dni ordenando "uno" de mayor a menor
w = Window.partitionBy("dni").orderBy(F.col("hoja").asc())

df_lista_2 = (
    df_lista_1
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [13]:

df_lista_2.show(4)

+--------+--------------------+---------+---------+----+--------------+-----------+------------------+-------------+--------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+-------------+----------------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+-----------+------------+
|     dni|              nombre|  celular|  agencia|hoja|   COLOR_FINAL|COD_USER_V3|           USER_V3|    PERFIL_RO| campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|   GRUPO_TASA|       TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA_1_SS|TASA_2_SS|TASA_3_SS|TASA_4_SS|TASA_5_SS|TASA_6_SS|TASA_7_SS|ALER

In [5]:
df_lista_pd=df_lista_2.toPandas()

In [6]:
df_lista_pd["agencia"] = df_lista_pd["agencia"].str.strip()

In [7]:
reemplazos = {
    "PC TACNA": "TACNA",
    "AREQ PAMPILLA": "AREQUIPA PAMPILLA",      # este realmente no cambia
    "AREQ CAYMA": "AREQUIPA CAYMA",
    "ENMANCIPACION": "EMANCIPACION",
    "SAN JUAN DE LURIG": "SAN JUAN DE LURIGANCHO",
    "TRUJ CENTRO": "TRUJILLO CENTRO",
    "TRUJ AMERICA": "TRUJILLO AMERICA",
    "PC HUANCAYO": "HUANCAYO",
    "PC HUARAZ": "HUARAZ",
    "PAITA": "SULLANA",
}

df_lista_pd["agencia"] = df_lista_pd["agencia"].replace(reemplazos)

In [8]:
query = f"""
select agencia_Formulario,agencia_correo  as agencia from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)

In [9]:
df_lista_pd_01=df_lista_pd.merge(df_Age,on='agencia',how='left')

In [10]:
df_lista_pd_01.loc[
    df_lista_pd_01["agencia_Formulario"].isna(),
    "agencia"
].unique()

array([], dtype=object)

In [11]:
query = f"""
	SELECT dni_cliente as dni,estado  as estdo_formulario, DATE(fecha_envio) as fecha_envio_formulario FROM Alice.prospectos_envio_alfin 
    where DATE(fecha_envio)>='2026-08-01'
"""
df_formulario_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT dni_cliente as dni,estado as estdo_correo,DATE(fecha_envio) as fecha_envio_correo FROM Alice.prospectos_correos_alfin 
    where DATE(fecha_envio)>='2026-08-01'
"""
df_correos_alfin = pd.read_sql(query, engine_mysql)


In [12]:
df_formulario_alfin["fecha_envio_formulario"] = pd.to_datetime(
    df_formulario_alfin["fecha_envio_formulario"]
)

df_formulario_alfin = (
    df_formulario_alfin
    .sort_values(
        "fecha_envio_formulario",
        ascending=False
    )
    .drop_duplicates(
        subset="dni",
        keep="first"
    )
)
df_correos_alfin["fecha_envio_correo"] = pd.to_datetime(
    df_correos_alfin["fecha_envio_correo"]
)

df_correos_alfin = (
    df_correos_alfin
    .sort_values(
        "fecha_envio_correo",
        ascending=False
    )
    .drop_duplicates(
        subset="dni",
        keep="first"
    )
)

In [13]:
df_lista_pd_02=df_lista_pd_01.merge(df_formulario_alfin,on='dni',how='left')
df_lista_pd_02=df_lista_pd_02.merge(df_correos_alfin,on='dni',how='left')



In [14]:
df_lista_pd_02 = df_lista_pd_02.drop_duplicates(
    subset="dni",
    keep="last"
)

In [15]:
import numpy as np
import pandas as pd

# Convertir a numérico
df_lista_pd_02["OFERTA_MAX"] = pd.to_numeric(df_lista_pd_02["OFERTA_MAX"], errors="coerce")

bins = [0, 5000, 10000, 15000, 20000, 25000, 30000, np.inf]

labels = [
    "01. [0 - 5,000)",
    "02. [5,000 - 10,000)",
    "03. [10,000 - 15,000)",
    "04. [15,000 - 20,000)",
    "05. [20,000 - 25,000)",
    "06. [25,000 - 30,000)",
    "07. >= 30,000"
]

df_lista_pd_02["RANGO_OFERTA"] = pd.cut(
    df_lista_pd_02["OFERTA_MAX"],
    bins=bins,
    labels=labels,
    right=False
)

In [16]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')

# df_target_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
# df_target_desembolso.rename(columns={'FECHA_DESEMBOLSOS': 'fecha_desembolso'}, inplace=True)
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['target'] = 1

filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','MONTO','ASESOR','CANALVENTA']].copy()

df_desembolso = df_target_desembolso[['DNI']].drop_duplicates().merge(
    df_fugas[['DNI']].drop_duplicates(),
    on=['DNI'],
    how='inner'
)

df_desembolso = df_target_desembolso.drop_duplicates().merge(
    df_fugas.drop_duplicates(),
    on=['DNI'],
    how='inner'
)
df_desembolso['target'] = df_desembolso['target'].fillna(0).astype(int)
df_desembolso['fugas'] = (df_desembolso['target'] == 0).astype(int)
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)

df_desembolso['dni_cliente'] = (
    df_desembolso['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)



In [17]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()




In [18]:

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


C:\Users\DATA\AppData\Local\Temp\ipykernel_18468\732878998.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [19]:
df_lista_pd_02 = df_lista_pd_02[
    (~df_lista_pd_02["dni"].isin(dni_retiro)) &
    (~df_lista_pd_02["dni"].isin(dni_desembolso)) &
    (~df_lista_pd_02["celular"].isin(cel_retiro))
].copy()

In [21]:
print(df_lista_pd_02.columns.tolist())

['dni', 'nombre', 'celular', 'agencia', 'hoja', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo', 'agencia_Formulario', 'estdo_formulario', 'fecha_envio_formulario', 'estdo_correo', 'fecha_envio_correo', 'RANGO_OFERTA']


In [20]:

ruta_archivo = os.path.join(ruta_csv, 'resumen_envio.csv')
df_lista_pd_02.to_csv(ruta_archivo,sep=';')

In [61]:
df_lista_pd_02.head(2)

,nombre,celular,agencia,hoja,COLOR_FINAL,COD_USER_V3,USER_V3,PERFIL_RO,campaña,OFERTA_MAX,...,FEN,PERFIL_ESPECIAL,TIPO,tipo_archivo,agencia_Formulario,estdo_formulario,fecha_envio_formulario,estdo_correo,fecha_envio_correo,RANGO_OFERTA
0,MELCHOR ANTONIO MERCADO RUIZ,950582801,PUCALLPA,1,NARANJA OSCURO,3,3. MES + PLD Peers,Aceptante I,SOLODNI,8000.0,...,None,None,PRESTALTOKE,campo,738334 - PUCALLPA,NaN,NaT,NaN,NaT,"02. [5,000 - 10,000)"
1,OSWALDO GONZALES CURINUQUI,939173565,TARAPOTO,1,VERDE OSCURO,3,3. MES + PLD Peers,Inconforme II,SOLODNI,14200.0,...,None,None,PRESTALTOKE,campo,733824 - TARAPOTO,NaN,NaT,NaN,NaT,"03. [10,000 - 15,000)"


In [ ]:
# df=df_subir.toPandas()
query = f"""
select agencia_Formulario,agencia_correo from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)
ruta_archivo = os.path.join(ruta_csv, 'SOOOOO.csv')
df_Age.to_csv(ruta_archivo, index=False)

'PC TACNA', 'AREQ PAMPILLA', 'ENMANCIPACION', 'SAN JUAN DE LURIG',
       'AREQ CAYMA', 'TRUJ CENTRO', 'TRUJ AMERICA', 'PC HUANCAYO',
       'PC HUARAZ', 'PAITA'

In [20]:
df_lista_pd_01[df_lista_pd_01['agencia_Formulario'].isna()].head()


,dni,nombre,celular,agencia,hoja,COLOR_FINAL,COD_USER_V3,USER_V3,PERFIL_RO,campaña,...,TASA_4_SS,TASA_5_SS,TASA_6_SS,TASA_7_SS,ALERTA_MAQUETA,FEN,PERFIL_ESPECIAL,TIPO,tipo_archivo,agencia_Formulario
45,00401050,JUAN CARLOS SUAREZ COHAILA,984495438,PC TACNA,1,VERDE CLARO,7,7. Peers,Aceptante II,SOLO CON DNI - ZONA COBERTURA,...,65,62,56,56,None,None,None,PRESTALTOKE,campo,NaN
46,00405457,MARIA ESTER AROS DE CARNERO,952079384,PC TACNA,1,NARANJA CLARO,6,6. MES B,Aceptante I,SOLODNI,...,110,110,110,110,None,None,None,PRESTALTOKE,campo,NaN
47,00418793,BLANCA ISABEL VASCONES DE PALZA,976563114,PC TACNA,1,VERDE OSCURO,8,8. Tarjetero Cash,Inconforme II,MUJER,...,91,91,91,91,None,None,None,PRESTALTOKE,campo,NaN
48,00426087,PATRICIA FERNANDA YESQUEN OTTONE,949200194,PC TACNA,1,NARANJA CLARO,14,14. Otros Bancarizados,Inconforme I,SOLODNI,...,110,110,110,110,None,None,None,PRESTALTOKE,campo,NaN
49,00429633,VALDIVIA GAMBOA WILBER RAFAEL,965659905,PC TACNA,1,AMARILLO CLARO,2,2. sunedu & sunarp B,Resistente II,SOLODNI,...,83,81,73,73,None,None,None,PRESTALTOKE,campo,NaN


In [3]:
filename='usar_01.csv'
df_subir=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

In [5]:
df_subir=df_subir.withColumnRenamed('dni_cliente','DNI')

In [4]:
df_base=df_base.drop( 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo')

In [5]:
filename='descartar!11.csv'
df_quitar_pues=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)


In [8]:
df_validar = df_validar.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )

)

In [9]:
df_subir=df_subir.join(df_validar,['DNI'],'inner')

In [10]:
df_subir=df_subir.dropDuplicates(['DNI'])

In [94]:


filename='desembolso_alfin.csv'
desembolso_quitar=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
desembolso_quitar=desembolso_quitar.withColumnRenamed('Número Documento','DNI')
desembolso_quitar.show(2)

+------+----------------+--------------+--------------+----------+-----------+----------------+------+----------------+-----------------+------------+--------------------+--------------------+------+--------+--------------+---------+------------+-----+------------------+-------------+-------------+-------------------+---------------+-----------+-----+-------------+----------+---+-----+-----+------+-----------+-------------+-----+------+--------------------+--------------------+------------+--------------------+----------+--------------+--------------+----------------------+------------+-----------------+---------------------+-------------+--------------------+-------------------+--------+---------+------------------+
|AñoMes|Fecha Desembolso|Canal Venta BT|Canal de Venta|    Región|CodSuc (BT)|   Sucursal (BT)|CodSuc|        Sucursal|Cod Empleado (BT)|Cod Empleado|          Nombre ANC|         Vendor (BT)|Vendor|     DNI|Cuenta Cliente|Operación|     Nombres|  Mda|Capital Solicitado|To

In [95]:
df_quitar_pues=df_quitar_pues.withColumnRenamed('dni_cliente','DNI')

In [ ]:
df_base_01

In [37]:
desembolso_quitar.count()

9254

In [ ]:


filename='BASE_CEL_TARGET_20260801.txt'
df_score=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)

filename='fomato_agendas_alfin_credicash_2026.csv'
formato_agendas=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='RetiroDefinitivo_BlackList.csv'
df_def_blacklist=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
filename='RetiroDeGestion_BlackList.csv'
df_blacklist=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
filename='RetiroDeGestion_Telefonos.csv'
df_retirogestion=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
filename='retiro_correo_alfin.csv'
df_retiro_correo=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

filename='agencia_alfin_cod.csv'
df_agencia_alfin=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)

print(df_base.columns)
print(df_score.columns)
print(df_validar.columns)
print(formato_agendas.columns)
print(df_def_blacklist.columns)
print(df_blacklist.columns)
print(df_retirogestion.columns)
print(df_retiro_correo.columns)
print(df_agencia_alfin.columns)

['PROPENSION_IC', 'DNI', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'tasa_minima', 'CUOTA', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'GRUPO_MONTO', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION']
['DNI', 'CET', 'CEL1', 'SCORE_TELEFONO']
['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSI

In [ ]:
['PROPENSION_IC', 'USER_V3', 'DNI', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'Campaña', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION']acc


In [59]:
df_base.dropDuplicates(['DNI']).count()

175755

In [7]:
df_quitar_pues.show(3)

+---+-----------+
|_c0|dni_cliente|
+---+-----------+
|  0|   48124718|
|  1|   41494656|
|  2|   41298318|
+---+-----------+
only showing top 3 rows


In [8]:
df_quitar_pues=df_quitar_pues.withColumnRenamed('dni_cliente','DNI')


In [9]:
df_base_01=df_base.join(df_validar,['DNI'],'inner')
df_base_01=df_base.join(df_quitar_pues,['DNI'],'left_anti')
df_base_01=df_base_01.dropDuplicates(['DNI'])

In [61]:
formato_agendas.show(3)

+-----------+--------------------+---------+--------------------+----------------+------------+-----+-------+
|dni_cliente|      nombre_cliente|  celular|         cod_agencia|agencia_atencion|fecha_visita|monto|color_1|
+-----------+--------------------+---------+--------------------+----------------+------------+-----+-------+
|    6757927|QUIROZ MONCADA SA...|987613889|738382 - JESUS MARIA|            NULL|        NULL|20000|   NULL|
|     426135|María asunta cáce...|969562424|   738013 - PC TACNA|            NULL|        NULL| 9900|   NULL|
|   22488348|Patricia Mabel Me...|962578360|    735996 - HUANUCO|            NULL|        NULL|16400|   NULL|
+-----------+--------------------+---------+--------------------+----------------+------------+-----+-------+
only showing top 3 rows


In [62]:
formato_agendas=formato_agendas.withColumnRenamed('dni_cliente','DNI')

In [63]:
formato_agendas1=formato_agendas.join(df_validar,['DNI'],'inner')
formato_agendas1=formato_agendas1.dropDuplicates(['DNI'])
formato_agendas1.count()

13035

In [ ]:
formato_agendas1=formato_agendas1

In [51]:
df=formato_agendas1.toPandas()

In [64]:
ruta_archivo = os.path.join(ruta_csv, 'consulta_campana_2.xlsx')
df.to_excel(ruta_archivo, index=False)

In [16]:
# df=df_subir.toPandas()
query = f"""
select agencia_Formulario,agencia_correo from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)
ruta_archivo = os.path.join(ruta_csv, 'SOOOOO.csv')
df_Age.to_csv(ruta_archivo, index=False)

In [79]:
print(df_base_01.columns)

['DNI', 'Agencia_comercial', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo', 'CET', 'CEL1', 'SCORE_TELEFONO']


In [10]:
df_base_01=df_base_01.join(df_score,['DNI'],'inner')

In [ ]:
['DNI', 'Agencia_comercial', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo', 'CET', 'CEL1', 'SCORE_TELEFONO']lo

In [ ]:
# grupo_tasa=['None', '', '', '', '', ]
# agencia_comercial=[ 'SAN MIGUEL', 'MIRAFLORES']

df_filtrado = df_base_01.filter(
    # (F.col('PROPENSION_DISTRIBUCION').isin('1','2','3'))&
    # (F.col('proveedor').isin(['INVENTARIO TARGET 2', 'CET TARGET 2', 'CET TARGET 0', 'INVENTARIO TARGET 1', 'INVENTARIO TARGET 0', ]))&
    # (F.col('user_v3').isin(user_v3))&

    # (

    (
        (F.col('USER_V3').isin('1. sunedu & sunarp A','2. sunedu & sunarp B','3. MES + PLD Peers'))&
        (F.col('TIPO_CLIENTE')=='INDEPENDIENTE')
    )
        
        (F.col('OFERTA_MAX')>=10000)&
        (F.col('lote')!='BOT')
    (F.col('retiro')=='ACTIVO'))

)



print(df_filtrado.count())
print(df_filtrado.columns)

In [44]:
formato_agendas.count()

22718

In [47]:
formato_agendas.dropDuplicates(['dni_cliente']).count()

22718

In [ ]:
df_base_telf=df_base_telf

In [39]:
df_base_01=df_base.filter(F.col('Agencia_comercial').isNotNull())
df_base_01=df_base_01.select('DNI','Agencia_comercial')

In [80]:
print([row['ACCION' ] for row in df_base.select('ACCION').distinct().collect()])


['NO CLIENTE INTERESADO OPERADOR', 'NO CLIENTE TICKET <= 5K ALTA PROPENSION', 'NO CLIENTE CONTACTADO OPERADOR', 'NO CLIENTE RESTO']


In [82]:
df_base=df_base.join(df_quitar_pues,['DNI'],'left_anti')

In [84]:
df_base.groupBy('ACCION') \
    .count() \
    .orderBy('ACCION') \
    .show(30,truncate=False)


+---------------------------------------+------+
|ACCION                                 |count |
+---------------------------------------+------+
|NO CLIENTE CONTACTADO OPERADOR         |10333 |
|NO CLIENTE INTERESADO OPERADOR         |3615  |
|NO CLIENTE RESTO                       |55485 |
|NO CLIENTE TICKET <= 5K ALTA PROPENSION|105902|
+---------------------------------------+------+



In [ ]:
df_base_02=df_base.join(df_validar,['DNI'],'inner')


In [11]:
# df_base_01=df_base.join(df_validar,['DNI'],'inner')
df_base_01=df_base_01.filter(F.col('ACCION')=='NO CLIENTE INTERESADO OPERADOR')
df_base_01=df_base_01.dropDuplicates(['DNI'])
df_base_01.count()

3615

In [12]:
df_base_03=df_base_01.filter(F.col('OFERTA_MAX')>=5000)
df_base_03.count()

{"ts": "2026-08-01 10:13:35.525", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `OFERTA_MAX` cannot be resolved. Did you mean one of the following? [`CET`, `CUOTA`, `FLG_AAHH`, `CEL1`, `DISTRITO`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o180.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `OFERTA_MAX` cannot be resolved. Did you mean one of the following? [`CET`, `CUOTA`, `FLG_AAHH`, `CEL1`, `DISTRITO`]. SQLSTATE: 42703;\n'Filter '`>=`('OFERTA_MAX, 5000)\n+- Deduplicate [DNI#19]\n   +- Filter (ACCION#70 = NO CLIENTE INTERESADO OPERADOR)\n      +- Pro

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `OFERTA_MAX` cannot be resolved. Did you mean one of the following? [`CET`, `CUOTA`, `FLG_AAHH`, `CEL1`, `DISTRITO`]. SQLSTATE: 42703;
'Filter '`>=`('OFERTA_MAX, 5000)
+- Deduplicate [DNI#19]
   +- Filter (ACCION#70 = NO CLIENTE INTERESADO OPERADOR)
      +- Project [DNI#19, PROPENSION_IC#17, X_APPATERNO#20, X_APMATERNO#21, X_NOMBRE#22, tasa_minima#24, CUOTA#26, TIPO_GEST#28, TIPO_CLIENTE_COMERCIAL#29, SALDO_DIFERENCIAL_REENG#31, TIPO_CLIENTE#32, DEPARTAMENTO#36, PROVINCIA#37, DISTRITO#38, SUCURSAL_COMERCIAL#39, Agencia_comercial#40, REGION_COMERCIAL#41, VARIACION_OFERTA_CAMPAÑA_ANTERIOR#42, VARIACION_TASA_CAMPAÑA_ANTERIOR#43, VAR_TASA_CREDITO_ANTERIOR#44, FLG_CET_6M#45, GRUPO_MONTO#48, Edad#57, RANGO_EDAD#58, RANGO_OFERTA#59, ... 13 more fields]
         +- Join Inner, (DNI#19 = DNI#346)
            :- Deduplicate [DNI#19]
            :  +- Project [DNI#19, PROPENSION_IC#17, X_AP

In [ ]:
print(df_base_02.columns)['DNI', 'PROPENSION_IC', 'USER_V3', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'Campaña', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'TIPO', 'tipo_archivo']ofe

['DNI', 'PROPENSION_IC', 'USER_V3', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'Campaña', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÑA_ANTERIOR', 'VARIACION_TASA_CAMPAÑA_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP', 'EXCLUSIVO', 'PILOTO_RETENCION', 'ACCION', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_A

In [ ]:

df_filtrado=df_filtrado.withColumnRenamed('NUMERO_DOCUMENTO','vendor_lead_code')
# df_filtrado=df_filtrado.withColumnRenamed('cl_telf1','phone_number')
# df_filtrado=df_filtrado.withColumnRenamed('NOMBRES','address1')

df_filtrado = df_filtrado.withColumn(
    "address1",
    F.concat_ws(
        " ",
        F.col("NOMBRES"),
        F.col("APELLIDO_PATERNO"),
        F.col("APELLIDO_MATERNO")
    )
)

df_filtrado = df_filtrado.withColumn(
    "security_phrase",
    F.concat_ws(
        " ",
        F.lit("Oferta:"),
        F.col("OFERTA_MAX")
    )
)
df_filtrado = df_filtrado.withColumn(
    "city",
    F.concat_ws(
        " ",
        F.lit("Departamento:"),
        F.col("DEPARTAMENTO")
    )
)
df_filtrado = df_filtrado.withColumn(
    "province",
    F.concat_ws(
        " ",
        F.lit("Distrito:"),
        F.col("DISTRITO")
    )
)

df_filtrado.count()

In [81]:
df_base.groupBy('ACCION') \
    .count() \
    .orderBy('ACCION') \
    .show(30)


+--------------------+------+
|              ACCION| count|
+--------------------+------+
|NO CLIENTE CONTAC...| 10394|
|NO CLIENTE INTERE...|  3956|
|    NO CLIENTE RESTO| 55503|
|NO CLIENTE TICKET...|105902|
+--------------------+------+



In [ ]:
['LOS OLIVOS', 'VILLA MARIA 2', 'MOSHOQUEQUE', 'PUENTE PIEDRA', 'COMAS', 'PUCALLPA', 'PISCO', 'TARAPOTO', 'CHINCHA', 'VENTANILLA', 'TUMBES', 'VILLA EL SALVADOR 2', 'PC TACNA', 'CAÑETE', 'HUANUCO', 'SANTA ANITA', 'JESUS MARIA', 'EMANCIPACION', 'CAJAMARCA', 'SAN MARTIN', 'SAN MIGUEL', 'MIRAFLORES', 'AREQUIPA CAYMA', 'SULLANA', 'AREQUIPA PAMPILLA', 'IQUITOS', 'ATE VITARTE', 'CHICLAYO BALTA', 'HUACHO', 'SAN JUAN DE MIRAFLORES', 'SAN JUAN DE LURIGANCHO', 'PC HUARAZ', 'PC HUANCAYO', 'CASTILLA', 'ICA', 'TRUJILLO CENTRO', 'CHIMBOTE', 'HUARAL', 'JULIACA 2', 'TRUJILLO AMERICA', 'CUSCO LA CULTURA', None]cañ

+--------+---------------+-----------+--------------------+-------------+-------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+-------------+----------------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+-----------+------------+
|     DNI|    COLOR_FINAL|COD_USER_V3|             USER_V3|    PERFIL_RO|campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|   GRUPO_TASA|       TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA_1_SS|TASA_2_SS|TASA_3_SS|TASA_4_SS|TASA_5_SS|TASA_6_SS|TASA_7_SS|ALERTA_MAQUETA| FEN|PERFIL_ESPECIAL|       TIPO|tipo_archivo|
+--------+---------------+----

In [48]:
formato_agendas=formato_agendas.withColumnRenamed('dni_cliente','DNI')

In [98]:

def completar_dni(df):
    return df.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )

df_base_01 = completar_dni(df_base_01)
df_validar = completar_dni(df_validar)
df_def_blacklist = completar_dni(df_def_blacklist)
df_blacklist = completar_dni(df_blacklist)
# df_retiro_correo = completar_dni(df_retiro_correo)
formato_agendas = completar_dni(formato_agendas)
df_quitar_pues = completar_dni(df_quitar_pues)
df_score = completar_dni(df_score)

{"ts": "2026-08-01 10:08:01.603", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `DNI` cannot be resolved. Did you mean one of the following? [`monto`, `celular`, `color_1`, `cod_agencia`, `dni_cliente`]. SQLSTATE: 42703", "context": {"file": "line 4 in cell [99]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1883.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `DNI` cannot be resolved. Did you mean one of the following? [`monto`, `celular`, `color_1`, `cod_agencia`, `dni_cliente`]. SQLSTATE: 42703;\n'Project [dni_cliente#10623, nombre_cliente#10624, celular#10625, cod_agencia#10626, agencia_atencion#10627, fecha_visita#10628, monto#10629, color_1#10630, 'lpad(cast('DNI

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `DNI` cannot be resolved. Did you mean one of the following? [`monto`, `celular`, `color_1`, `cod_agencia`, `dni_cliente`]. SQLSTATE: 42703;
'Project [dni_cliente#10623, nombre_cliente#10624, celular#10625, cod_agencia#10626, agencia_atencion#10627, fecha_visita#10628, monto#10629, color_1#10630, 'lpad(cast('DNI as string), 8, 0) AS DNI#10739]
+- Relation [dni_cliente#10623,nombre_cliente#10624,celular#10625,cod_agencia#10626,agencia_atencion#10627,fecha_visita#10628,monto#10629,color_1#10630] csv


In [7]:
df_base=df_base.join(df_def_blacklist,['DNI'],'leftanti')
df_base=df_base.join(df_blacklist,['DNI'],'leftanti')
df_base=df_base.join(df_retiro_correo,['DNI'],'leftanti')

In [ ]:

df_base = df_base.withColumnRenamed('X_APPATERNO','APELLIDO_PATERNO')
df_base = df_base.withColumnRenamed('X_APMATERNO','APELLIDO_MATERNO')
df_base = df_base.withColumnRenamed('CampaÃ±a','campania')
df_base = df_base.withColumnRenamed('X_NOMBRE','NOMBRES')
df_base = df_base.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_base = df_base.withColumnRenamed('SUCURSAL_COMERCIAL','sucursal_comercial')
df_base = df_base.withColumnRenamed('FLAG_DEUDA_V_OFERTA','flag_deuda_v_oferta')
df_base = df_base.withColumnRenamed('REGION_COMERCIAL','Region_comercial')
df_base = df_base.withColumnRenamed('TIPO_CLIENTE_COMERCIAL','tipo_cliente_riegos')
df_base = df_base.withColumn("Campana", F.lit("202606"))
df_base = df_base.withColumn("lote", F.lit("BASE 2026-06-27"))
df_base = df_base.withColumn("cl_base", F.lit("Junio 2026"))
df_base = df_base.withColumn("cl_carga", F.lit("2026-07-27"))
df_base = df_base.withColumn("cl_estado", F.lit("1"))
df_base = df_base.withColumn("estado", F.lit("ACTIVO"))


In [13]:
query = f"""
    SELECT * FROM DANTALION.[dbo].Base_Maestra_Alfin_bk
    where fecha_envio>='2026-07-01'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
df_formato.show(2)

+--------+----------------+-----------+----------------+----------------+--------+------------+------------+----------------+--------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+-------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+--------------------+---------+---------------------+-----+------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+----------+---------+---------+---------+----------+---------+------------+-------------+-------+------------------------+-

In [14]:
from pyspark.sql import functions as F

exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [15]:
en_formato=set(df_formato.columns)
en_base=set(df_base.columns)

In [12]:
print(en_formato-en_base)
print(en_base-en_formato)

{'PROPENSION', 'cl_telf7', 'FLAG_REENG', 'cl_mes', 'Desgravamen_12M', 'Validador_Telefono', 'lote', 'CUOTA_36M', 'NUEVOS_9M', 'NUM_ENRIQUECIDO', 'cl_celular', 'Deuda_1', 'ID_CLIENTE', 'SBI', 'cl_telf5', 'APELLIDO_PATERNO', 'cl_accion', 'campania', 'Desgravamen', 'AÑO_DURACION_BASE', 'Prioridad', 'APELLIDO_MATERNO', 'incremento_monto_riesgos', 'cl_tiempo', 'REP1', 'OFERTA_REEN', 'Entidad_2', 'CUOTA_12M', 'cl_area', 'COD_BD', 'cl_telefono', 'RANGO_EDAD2', 'RETIRO', 'Desgravamen_18M', 'TIENDA', 'TIPO_CONTACTO', 'cl_hits', 'FLAT2', 'REP2', 'CUOTA_24M', 'SEGMENTO_USER', 'cl_turno', 'cl_gestor', 'PROMOCION2', 'sucursal_comercial', 'cl_telf10', 'TIPO_BD', 'CLIENTE_NUEVO', 'RETIRO_GEST', 'OfertaMaximaSinSeguro', 'FECHA_SOL', 'USUARIO', 'cl_carga', 'PERFIL_GLOBAL', 'Desgravamen_24M', 'Oferta_Minima_Paperless', 'Oferta_18M', 'cl_telf1', 'Tasa_18M', 'cl_fecha_llamar', 'Entidad_1', 'cl_telf2', 'cl_hora_gestion', 'NUEVOS_12M', 'marca3', 'NOMBRES', 'color', 'SUCURSAL', 'MONTO_DESEMBOLSADO', 'cl_base

In [ ]:
query = f"""
    SELECT *
        FROM (
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono as cl_telf1,  1 as desembolso,1 as ordern 
            from VALENTINA.dbo.alfcc_ventas a
            inner join cronox.dbo.ref_alfin_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
            UNION ALL
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono as cl_telf1,  1 as desembolso,  2 as orden
            from VALENTINA.dbo.alfin_ventas a
            inner join cronox.dbo.ref_alfin_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
        ) t
        """
df_desembolso=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_desembolso.count()

In [ ]:
query = f"""
    SELECT *
    FROM (
        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 6 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesCencosudTc] a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 4 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesDiners a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 8 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesEfectiva a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.DNI_CLIENTE AS NUMERO_DOCUMENTO,a.Telefono_Llamado AS TELEFONO,a.ESTADOS AS TIPO_GESTION, a.Descripcion AS GESTION,a.FECHA_ENVIO, 7 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionAgentesEfectivaN a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.DNI_CLIENTE = b.NUMERO_DOCUMENTO
        WHERE a.Telefono_Llamado IS NOT NULL

        UNION ALL

        SELECT a.CODDOC AS NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 5 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesCencoPP a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.CODDOC = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.fecha_llamada AS FECHA_ENVIO, 3 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesDinerstc a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL
    ) t
    """
df_maeba=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)


query = f"""
    SELECT *
        FROM (
            select 
            a.dni_cliente as NUMERO_DOCUMENTO,  
            a.celular as TELEFONO,
            a.Fecha as FECHA_ENVIO,
            b.nivel_1 AS TIPO_GESTION,
            b.nivel_2 AS GESTION ,2 as peso 
            FROM VALENTINA.dbo.alfin_gestion a
            inner join cronox.dbo.ref_alfin_dni C
            on a.dni_cliente COLLATE Modern_Spanish_CI_AS=C.NUMERO_DOCUMENTO   
            LEFT JOIN VALENTINA.dbo.alfin_tipificaciones B 
            ON A.Tipificacion = B.id_banco
            where a.fecha<'2026-06-01'
            UNION ALL
            select 
            a.dni_cliente as NUMERO_DOCUMENTO,  
            a.celular as TELEFONO,
            a.Fecha as FECHA_ENVIO,
            b.nivel_1 AS TIPO_GESTION,
            b.nivel_2 AS GESTION, 1 as peso 
            FROM VALENTINA.dbo.alfcc_gestion a
            inner join cronox.dbo.ref_alfin_dni C
            on a.dni_cliente COLLATE Modern_Spanish_CI_AS=C.NUMERO_DOCUMENTO   
            LEFT JOIN VALENTINA.dbo.alfcc_tipificaciones B 
            ON A.Tipificacion = B.id_banco
            where a.fecha<'2026-06-01'
        ) t
        """
df_valentina=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_cet=df_maeba.unionByName(df_valentina)



window_spec = Window.partitionBy("NUMERO_DOCUMENTO").orderBy(F.col("peso").asc(),F.col("FECHA_ENVIO").desc())
df_cet = df_cet.withColumn("ref_01", row_number().over(window_spec))
df_cet = df_cet.filter(col("ref_01") == 1).drop('ref_01','peso')

In [ ]:
df_desembolso = df_desembolso.withColumn(
    "dif_meses",
    F.floor(
        F.months_between(
            F.to_date(F.lit('2026-05-01')),
            F.col("fecha_llamada")
        )
    )
)

df_desembolso=df_desembolso.withColumn('marca',when(F.col('dif_meses').isin(0,1),'INVENTARIO TARGET 0')
                                                .when(F.col('dif_meses').isin(2,3,4),'INVENTARIO TARGET 1')
                                                .otherwise(F.lit('INVENTARIO TARGET 2'))
)

df_gestion=df_gestion.filter(F.col('nivel_2').isin(nivel_2))
df_gestion=df_gestion.filter(F.col('nivel_1').isin('CONTACTO EFECTIVO CON TITULAR'))
df_gestion = df_gestion.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )

)

df_gestion=df_gestion.withColumn('marca',when(F.col('dif_meses').isin(0,1),'CET TARGET 0')
                                                .when(F.col('dif_meses').isin(2,3,4),'CET TARGET 1')
                                                .otherwise(F.lit('CET TARGET 2'))
)

In [ ]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)
